In [1]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder.appName("SmartCity")
  .master("spark://spark:7077")
  .config("spark.driver.host", "jupyter")
  .config("spark.driver.bindAddress", "0.0.0.0")
  .config("spark.executor.memory", "2g")
  .config("spark.cores.max", "6")
  .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3")
  .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
  .getOrCreate())
print(spark.version, spark.sparkContext.master)

3.5.0 spark://spark:7077


In [2]:
df = spark.read.csv("hdfs://namenode:9000/historico/datos_limpios.csv", header=True, inferSchema=True)
print(df.count(), df.rdd.getNumPartitions())

1062858 6


In [3]:
from pyspark.sql.functions import count, desc

print("=== 1. SENSORES MÁS ACTIVOS ===")
df.groupBy("sensor_nombre") \
  .agg(count("*").alias("total_lecturas")) \
  .orderBy(desc("total_lecturas")) \
  .show(10, truncate=False)

=== 1. SENSORES MÁS ACTIVOS ===
+----------------+--------------+
|sensor_nombre   |total_lecturas|
+----------------+--------------+
|CDMX_10699_35646|5000          |
|CDMX_10534_34945|5000          |
|CDMX_10658_35473|5000          |
|CDMX_10699_35639|5000          |
|CDMX_10648_35433|5000          |
|CDMX_10534_34944|5000          |
|CDMX_10658_35475|5000          |
|CDMX_10534_34943|5000          |
|CDMX_10558_35037|5000          |
|CDMX_10658_35484|5000          |
+----------------+--------------+
only showing top 10 rows



In [4]:
from pyspark.sql.functions import avg, round as spark_round

print("=== 2. DISTRIBUCIÓN GEOGRÁFICA POR ZONA ===")
df.groupBy("zona") \
  .agg(
      spark_round(avg("latitud"), 4).alias("lat_promedio"),
      spark_round(avg("longitud"), 4).alias("lon_promedio"),
      count("*").alias("total_eventos")
  ) \
  .orderBy(desc("total_eventos")) \
  .show(10, truncate=False)

=== 2. DISTRIBUCIÓN GEOGRÁFICA POR ZONA ===
+------------------+------------+------------+-------------+
|zona              |lat_promedio|lon_promedio|total_eventos|
+------------------+------------+------------+-------------+
|Suburban North    |40.7156     |-74.0185    |53040        |
|Pedregal          |19.3252     |-99.2041    |50000        |
|Camarones         |19.4685     |-99.1698    |50000        |
|Tlalnepantla      |19.5291     |-99.2046    |50000        |
|San Agustín       |19.533      |-99.0304    |49999        |
|Merced            |19.4247     |-99.1196    |49999        |
|UAM Iztapalapa    |19.3608     |-99.0738    |49998        |
|Financial District|40.7249     |-73.996     |48960        |
|Nezahualcóyotl    |19.393      |-99.0282    |45000        |
|Downtown Core     |40.7272     |-74.0023    |44880        |
+------------------+------------+------------+-------------+
only showing top 10 rows



In [5]:
from pyspark.sql.functions import year, month, avg, round as spark_round

print("=== 3. EVOLUCIÓN TEMPORAL (por año-mes) ===")
df.withColumn("anio", year("fecha")) \
  .withColumn("mes", month("fecha")) \
  .groupBy("anio", "mes") \
  .agg(
      spark_round(avg("pm25"), 2).alias("pm25_promedio"),
      spark_round(avg("carros"), 2).alias("carros_promedio"),
      count("*").alias("total")
  ) \
  .orderBy("anio", "mes") \
  .show(50, truncate=False)

=== 3. EVOLUCIÓN TEMPORAL (por año-mes) ===
+----+---+-------------+---------------+-----+
|anio|mes|pm25_promedio|carros_promedio|total|
+----+---+-------------+---------------+-----+
|2016|3  |19.83        |NULL           |8597 |
|2016|4  |30.34        |NULL           |39571|
|2016|5  |32.04        |NULL           |49666|
|2016|6  |22.2         |NULL           |48587|
|2016|7  |19.0         |NULL           |51084|
|2016|8  |18.31        |NULL           |45054|
|2016|9  |17.24        |NULL           |44649|
|2016|10 |15.91        |NULL           |47481|
|2016|11 |24.38        |NULL           |42399|
|2016|12 |21.69        |NULL           |10490|
|2017|1  |32.61        |NULL           |1697 |
|2017|2  |NULL         |NULL           |935  |
|2017|3  |NULL         |NULL           |753  |
|2017|4  |NULL         |NULL           |294  |
|2017|5  |NULL         |NULL           |575  |
|2017|6  |NULL         |NULL           |629  |
|2017|7  |NULL         |NULL           |410  |
|2017|8  |NULL  

In [6]:
from pyspark.sql.functions import col

print("=== 4. CORRELACIÓN TRÁFICO ↔ CONTAMINACIÓN ===")

df_corr = df.filter(
    col("carros").isNotNull() & col("aqi").isNotNull()
)
print(f"Filas válidas: {df_corr.count()}")

corr_aqi = df_corr.stat.corr("carros", "aqi")
print(f"Correlación carros vs AQI: {corr_aqi:.4f}")

df_corr2 = df.filter(
    col("carros").isNotNull() & col("emission_estimate").isNotNull()
)
corr_em = df_corr2.stat.corr("carros", "emission_estimate")
print(f"Correlación carros vs emisiones: {corr_em:.4f}")

corr_cong = df_corr.stat.corr("carros", "congestion_score")
print(f"Correlación carros vs congestión: {corr_cong:.4f}")

def interpretar(c):
    if abs(c) > 0.7: return "FUERTE"
    elif abs(c) > 0.3: return "MODERADA"
    else: return "DÉBIL"

print(f"\nInterpretación:")
print(f"  carros ↔ AQI: {interpretar(corr_aqi)}")
print(f"  carros ↔ emisiones: {interpretar(corr_em)}")
print(f"  carros ↔ congestión: {interpretar(corr_cong)}")

=== 4. CORRELACIÓN TRÁFICO ↔ CONTAMINACIÓN ===
Filas válidas: 204000
Correlación carros vs AQI: 0.0324
Correlación carros vs emisiones: 0.9253
Correlación carros vs congestión: 0.8847

Interpretación:
  carros ↔ AQI: DÉBIL
  carros ↔ emisiones: FUERTE
  carros ↔ congestión: FUERTE


In [7]:
from pyspark.sql.functions import avg, count, round as spark_round, desc

print("=== 5. RANKING DE ZONAS POR ACTIVIDAD ===")
df.groupBy("zona") \
  .agg(
      count("*").alias("total_lecturas"),
      spark_round(avg("pm25"), 2).alias("pm25_prom"),
      spark_round(avg("carros"), 2).alias("carros_prom")
  ) \
  .orderBy(desc("total_lecturas")) \
  .show(20, truncate=False)

=== 5. RANKING DE ZONAS POR ACTIVIDAD ===
+----------------------------------+--------------+---------+-----------+
|zona                              |total_lecturas|pm25_prom|carros_prom|
+----------------------------------+--------------+---------+-----------+
|Suburban North                    |53040         |NULL     |622.63     |
|Camarones                         |50000         |23.2     |NULL       |
|Pedregal                          |50000         |17.86    |NULL       |
|Tlalnepantla                      |50000         |23.33    |NULL       |
|San Agustín                       |49999         |21.69    |NULL       |
|Merced                            |49999         |25.45    |NULL       |
|UAM Iztapalapa                    |49998         |22.83    |NULL       |
|Financial District                |48960         |NULL     |895.61     |
|Nezahualcóyotl                    |45000         |21.18    |NULL       |
|Downtown Core                     |44880         |NULL     |838.08   

In [8]:
from pyspark.sql.functions import hour, avg, round as spark_round

print("=== 6. TENDENCIAS POR HORA DEL DÍA ===")
df.withColumn("hora", hour("fecha")) \
  .groupBy("hora") \
  .agg(
      spark_round(avg("carros"), 2).alias("carros_prom"),
      spark_round(avg("pm25"), 2).alias("pm25_prom"),
      spark_round(avg("congestion_score"), 2).alias("congestion_prom")
  ) \
  .orderBy("hora") \
  .show(24, truncate=False)

=== 6. TENDENCIAS POR HORA DEL DÍA ===
+----+-----------+---------+---------------+
|hora|carros_prom|pm25_prom|congestion_prom|
+----+-----------+---------+---------------+
|0   |176.97     |20.22    |15.47          |
|1   |177.17     |19.78    |15.4           |
|2   |176.97     |19.21    |15.52          |
|3   |176.86     |18.73    |15.49          |
|4   |177.07     |18.53    |15.46          |
|5   |176.87     |18.73    |15.44          |
|6   |177.23     |18.94    |15.5           |
|7   |703.43     |18.76    |60.52          |
|8   |1359.09    |19.6     |88.64          |
|9   |1884.95    |19.98    |95.51          |
|10  |1531.28    |20.11    |98.12          |
|11  |1705.32    |20.68    |99.47          |
|12  |1761.13    |21.05    |99.8           |
|13  |1705.26    |22.06    |99.48          |
|14  |1534.75    |23.61    |98.13          |
|15  |1252.15    |25.28    |95.19          |
|16  |1359.14    |27.18    |88.64          |
|17  |703.34     |29.29    |60.46          |
|18  |271.75    

In [9]:
from pyspark.sql.functions import avg, stddev, col

print("=== 7. DETECCIÓN DE ANOMALÍAS EN PM2.5 ===")
stats = df.agg(
    avg("pm25").alias("media"),
    stddev("pm25").alias("desv")
).collect()[0]

media = stats["media"]
desv = stats["desv"]
limite = media + 3 * desv

print(f"Media pm25: {media:.2f}")
print(f"Desviación: {desv:.2f}")
print(f"Límite: {limite:.2f}")

anomalias = df.filter(col("pm25") > limite)
print(f"Anomalías: {anomalias.count()}")
anomalias.select("sensor_nombre", "zona", "pm25", "fecha").show(10, truncate=False)

=== 7. DETECCIÓN DE ANOMALÍAS EN PM2.5 ===
Media pm25: 22.41
Desviación: 13.90
Límite: 64.12
Anomalías: 1467
+------------------+----------------------------------+----+-------------------+
|sensor_nombre     |zona                              |pm25|fecha              |
+------------------+----------------------------------+----+-------------------+
|CDMX_10534_34941  |Centro de Ciencias de la Atmósfera|65.0|2021-02-19 18:00:00|
|CDMX_10534_34941  |Centro de Ciencias de la Atmósfera|67.0|2021-03-13 18:00:00|
|CDMX_10632_6201284|Gustavo A. Madero                 |73.0|2024-10-25 19:00:00|
|CDMX_10632_6201284|Gustavo A. Madero                 |65.0|2024-10-25 20:00:00|
|CDMX_10632_6201284|Gustavo A. Madero                 |71.0|2024-10-25 21:00:00|
|CDMX_10632_6201284|Gustavo A. Madero                 |73.0|2024-10-30 20:00:00|
|CDMX_10632_6201284|Gustavo A. Madero                 |65.0|2025-01-15 18:00:00|
|CDMX_10632_6201284|Gustavo A. Madero                 |77.0|2025-01-15 19:00:00|


In [10]:
from pyspark.sql.functions import when, count, avg, round as spark_round, col

print("=== 8. AGRUPAMIENTO POR NIVEL DE CONTAMINACIÓN ===")
df.withColumn("nivel_pm25",
    when(col("pm25") < 12, "Bajo")
    .when(col("pm25") < 35, "Moderado")
    .when(col("pm25") < 55, "Alto")
    .otherwise("Muy Alto")
) \
.groupBy("nivel_pm25") \
.agg(
    count("*").alias("total"),
    spark_round(avg("pm25"), 2).alias("pm25_promedio"),
    spark_round(avg("carros"), 2).alias("carros_promedio")
) \
.orderBy("nivel_pm25") \
.show(truncate=False)

=== 8. AGRUPAMIENTO POR NIVEL DE CONTAMINACIÓN ===
+----------+------+-------------+---------------+
|nivel_pm25|total |pm25_promedio|carros_promedio|
+----------+------+-------------+---------------+
|Alto      |21200 |41.77        |NULL           |
|Bajo      |33491 |7.16         |NULL           |
|Moderado  |86715 |21.72        |NULL           |
|Muy Alto  |921452|66.28        |749.59         |
+----------+------+-------------+---------------+



In [11]:
from pyspark.sql.functions import year, avg, count, round as spark_round

print("=== 9. COMPARACIÓN HISTÓRICA POR AÑO ===")
df.withColumn("anio", year("fecha")) \
  .groupBy("anio") \
  .agg(
      count("*").alias("total_registros"),
      spark_round(avg("pm25"), 2).alias("pm25_prom"),
      spark_round(avg("carros"), 2).alias("carros_prom"),
      spark_round(avg("aqi"), 2).alias("aqi_prom")
  ) \
  .orderBy("anio") \
  .show(truncate=False)

=== 9. COMPARACIÓN HISTÓRICA POR AÑO ===
+----+---------------+---------+-----------+--------+
|anio|total_registros|pm25_prom|carros_prom|aqi_prom|
+----+---------------+---------+-----------+--------+
|2016|387578         |22.47    |NULL       |NULL    |
|2017|6401           |32.61    |NULL       |NULL    |
|2018|10051          |21.4     |NULL       |NULL    |
|2019|9482           |16.25    |NULL       |NULL    |
|2020|122869         |20.67    |NULL       |NULL    |
|2021|180774         |21.3     |NULL       |NULL    |
|2022|2142           |15.13    |NULL       |NULL    |
|2023|86397          |21.87    |NULL       |NULL    |
|2024|20018          |28.67    |NULL       |NULL    |
|2025|28215          |25.2     |NULL       |NULL    |
|2026|208931         |31.79    |749.59     |72.52   |
+----+---------------+---------+-----------+--------+



In [12]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, row_number, col, desc, avg

print("=== WINDOW FUNCTION: Ranking de zonas por contaminación ===")

ventana = Window.orderBy(desc("pm25_prom"))

resultado_window = df.filter(col("pm25").isNotNull()) \
    .groupBy("zona") \
    .agg(avg("pm25").alias("pm25_prom")) \
    .withColumn("rank", rank().over(ventana)) \
    .withColumn("dense_rank", dense_rank().over(ventana)) \
    .withColumn("row_number", row_number().over(ventana)) \
    .orderBy("rank")

resultado_window.show(15, truncate=False)

=== WINDOW FUNCTION: Ranking de zonas por contaminación ===
+----------------------------------------+------------------+----+----------+----------+
|zona                                    |pm25_prom         |rank|dense_rank|row_number|
+----------------------------------------+------------------+----+----------+----------+
|Mexico City                             |32.46793939399999 |1   |1         |1         |
|National Autonomous University of Mexico|30.00880358000003 |2   |2         |2         |
|Rincon del Bosque, CDMX                 |28.348277899999935|3   |3         |3         |
|Merced                                  |25.45184518451845 |4   |4         |4         |
|Santiago Acahualtepec                   |25.0954           |5   |5         |5         |
|Xalostoc                                |24.436            |6   |6         |6         |
|Tlalnepantla                            |23.3307           |7   |7         |7         |
|Camarones                               |23.1966 

In [13]:
from pyspark.sql.functions import col

print("=== JOIN: Medicion + Sensor + Zona ===")

df_sensores = spark.read.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/smartcity") \
    .option("dbtable", "sensor") \
    .option("user", "admin") \
    .option("password", "admin123") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_zonas = spark.read.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/smartcity") \
    .option("dbtable", "zona") \
    .option("user", "admin") \
    .option("password", "admin123") \
    .option("driver", "org.postgresql.Driver") \
    .load()

df_med_sensor = df.join(df_sensores, df["sensor_nombre"] == df_sensores["nombre"], how="inner")
df_completo = df_med_sensor.join(df_zonas, df_med_sensor["id_zona"] == df_zonas["id_zona"], how="inner")

print(f"Total filas después del JOIN: {df_completo.count()}")
df_completo.select("sensor_nombre", "nombre_zona", "pm25", "carros").show(10, truncate=False)

=== JOIN: Medicion + Sensor + Zona ===
Total filas después del JOIN: 1062858
+-------------+------------------+----+------+
|sensor_nombre|nombre_zona       |pm25|carros|
+-------------+------------------+----+------+
|INT_029      |Financial District|NULL|192.0 |
|INT_029      |Financial District|NULL|157.0 |
|INT_029      |Financial District|NULL|116.0 |
|INT_029      |Financial District|NULL|182.0 |
|INT_029      |Financial District|NULL|169.0 |
|INT_029      |Financial District|NULL|140.0 |
|INT_029      |Financial District|NULL|159.0 |
|INT_029      |Financial District|NULL|689.0 |
|INT_029      |Financial District|NULL|1277.0|
|INT_029      |Financial District|NULL|1722.0|
+-------------+------------------+----+------+
only showing top 10 rows



In [14]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql.functions import hour, col

print("=== ANÁLISIS PREDICTIVO: Regresión Lineal ===")

df_pred = df.filter(
    col("carros").isNotNull() & 
    col("temperatura").isNotNull() & 
    col("fecha").isNotNull()
).withColumn("hora", hour("fecha"))

df_pred = df_pred.select("hora", "temperatura", "carros").dropna()
print(f"Filas para entrenamiento: {df_pred.count()}")

assembler = VectorAssembler(
    inputCols=["hora", "temperatura"],
    outputCol="features"
)
df_features = assembler.transform(df_pred)

train, test = df_features.randomSplit([0.8, 0.2], seed=42)

lr = LinearRegression(
    featuresCol="features",
    labelCol="carros",
    maxIter=10,
    regParam=0.1
)
modelo = lr.fit(train)

print(f"\nCoeficientes: {modelo.coefficients}")
print(f"Intercepto: {modelo.intercept}")
print(f"R²: {modelo.summary.r2:.4f}")
print(f"RMSE: {modelo.summary.rootMeanSquaredError:.4f}")

=== ANÁLISIS PREDICTIVO: Regresión Lineal ===
Filas para entrenamiento: 204000

Coeficientes: [15.702172203998892,13.130390722471873]
Intercepto: 371.8569335158657
R²: 0.0096
RMSE: 750.8075


In [15]:
from pyspark.sql.functions import avg, count, round as spark_round

resultado_final = df.groupBy("zona") \
  .agg(
      count("*").alias("total_lecturas"),
      spark_round(avg("pm25"), 2).alias("pm25_prom"),
      spark_round(avg("carros"), 2).alias("carros_prom"),
      spark_round(avg("congestion_score"), 2).alias("congestion_prom")
  )

# Guardar en PostgreSQL
resultado_final.write.format("jdbc") \
    .option("url", "jdbc:postgresql://postgres:5432/smartcity") \
    .option("dbtable", "resultado_spark_zonas") \
    .option("user", "admin") \
    .option("password", "admin123") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()
print("✅ Guardado en PostgreSQL: resultado_spark_zonas")

# Guardar en HDFS
resultado_final.write.mode("overwrite") \
    .parquet("hdfs://namenode:9000/resultados/zonas_analisis")
print("✅ Guardado en HDFS: /resultados/zonas_analisis")

resultado_final.show(50, truncate=False)

✅ Guardado en PostgreSQL: resultado_spark_zonas
✅ Guardado en HDFS: /resultados/zonas_analisis
+----------------------------------------+--------------+---------+-----------+---------------+
|zona                                    |total_lecturas|pm25_prom|carros_prom|congestion_prom|
+----------------------------------------+--------------+---------+-----------+---------------+
|Financial District                      |48960         |NULL     |895.61     |52.45          |
|Suburban North                          |53040         |NULL     |622.63     |47.94          |
|Tech Park                               |22440         |NULL     |702.5      |47.88          |
|Downtown Core                           |44880         |NULL     |838.08     |52.48          |
|Residential West                        |18360         |NULL     |644.05     |47.93          |
|Industrial East                         |16320         |NULL     |664.24     |47.87          |
|Atizapán                                

In [16]:
df.createOrReplaceTempView("lecturas")
spark.sql("""
WITH por_zona AS (
  SELECT zona, COUNT(*) total, ROUND(AVG(carros),2) carros_prom, ROUND(AVG(pm25),2) pm25_prom
  FROM lecturas GROUP BY zona)
SELECT *, RANK() OVER (ORDER BY total DESC) ranking FROM por_zona
""").show(truncate=False)

spark.sql("""
SELECT HOUR(fecha) hora, ROUND(AVG(carros),1) carros, ROUND(AVG(congestion_score),1) congestion
FROM lecturas WHERE carros IS NOT NULL GROUP BY HOUR(fecha) ORDER BY hora
""").show(24)

+----------------------------------+-----+-----------+---------+-------+
|zona                              |total|carros_prom|pm25_prom|ranking|
+----------------------------------+-----+-----------+---------+-------+
|Suburban North                    |53040|622.63     |NULL     |1      |
|Camarones                         |50000|NULL       |23.2     |2      |
|Pedregal                          |50000|NULL       |17.86    |2      |
|Tlalnepantla                      |50000|NULL       |23.33    |2      |
|San Agustín                       |49999|NULL       |21.69    |5      |
|Merced                            |49999|NULL       |25.45    |5      |
|UAM Iztapalapa                    |49998|NULL       |22.83    |7      |
|Financial District                |48960|895.61     |NULL     |8      |
|Nezahualcóyotl                    |45000|NULL       |21.18    |9      |
|Downtown Core                     |44880|838.08     |NULL     |10     |
|Atizapán                          |40000|NULL     

In [17]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
mes = (df.filter(col("carros").isNotNull())
  .withColumn("mes", date_format("fecha","yyyy-MM"))
  .groupBy("zona","mes").agg(avg("carros").alias("carros_prom")))
w = Window.partitionBy("zona").orderBy("mes")
mes = mes.withColumn("var_pct", round((col("carros_prom")/lag("carros_prom").over(w)-1)*100,2))
mes.orderBy("zona","mes").show(30)

+------------------+-------+-----------------+-------+
|              zona|    mes|      carros_prom|var_pct|
+------------------+-------+-----------------+-------+
|     Downtown Core|2026-01|835.7797531769306|   NULL|
|     Downtown Core|2026-02|837.0482954545455|   0.15|
|     Downtown Core|2026-03|841.9214015151515|   0.58|
|Financial District|2026-01|892.9963597670251|   NULL|
|Financial District|2026-02|894.5130828373016|   0.17|
|Financial District|2026-03|899.8934294871794|    0.6|
|   Industrial East|2026-01|662.4408602150538|   NULL|
|   Industrial East|2026-02|663.6891741071429|   0.19|
|   Industrial East|2026-03|666.9681490384615|   0.49|
|  Residential West|2026-01|641.2547789725209|   NULL|
|  Residential West|2026-02|642.4573412698413|   0.19|
|  Residential West|2026-03|649.1102207977208|   1.04|
|    Suburban North|2026-01|620.5448200992556|   NULL|
|    Suburban North|2026-02|620.8895947802198|   0.06|
|    Suburban North|2026-03|627.0050542406311|   0.98|
|         

In [18]:
ws = Window.partitionBy("sensor_nombre")
an = (df.filter(col("carros").isNotNull())
  .withColumn("z", (col("carros")-avg("carros").over(ws))/stddev("carros").over(ws))
  .filter(abs(col("z"))>3))
print(an.count()); an.select("sensor_nombre","zona","fecha","carros","z").show(10)

66
+-------------+---------------+-------------------+------+------------------+
|sensor_nombre|           zona|              fecha|carros|                 z|
+-------------+---------------+-------------------+------+------------------+
|      INT_006|Industrial East|2026-02-04 09:00:00|2809.0|  3.06348548582599|
|      INT_007| Suburban North|2026-03-02 09:00:00|1869.0|3.0306190473460832|
|      INT_019| Suburban North|2026-01-21 09:00:00|1879.0| 3.073160345257628|
|      INT_019| Suburban North|2026-03-05 09:00:00|1906.0|3.1326367090289398|
|      INT_033|      Tech Park|2026-01-05 09:00:00|1876.0|3.0695058140399243|
|      INT_033|      Tech Park|2026-02-04 09:00:00|1878.0|3.0739138068797893|
|      INT_036|Industrial East|2026-01-06 09:00:00|2821.0|3.0593038591081982|
|      INT_036|Industrial East|2026-01-28 09:00:00|2785.0|3.0066822374594997|
|      INT_036|Industrial East|2026-02-26 09:00:00|2808.0|3.0403016068461683|
|      INT_041| Suburban North|2026-01-02 09:00:00|2824.0|3.0

In [19]:
from pyspark.ml.evaluation import RegressionEvaluator
pred = modelo.transform(test)
ev = RegressionEvaluator(labelCol="carros", predictionCol="prediction")
print("RMSE test:", ev.setMetricName("rmse").evaluate(pred))
print("R2 test:", ev.setMetricName("r2").evaluate(pred))

RMSE test: 751.0304627268968
R2 test: 0.010173205932056062


In [20]:
dj = spark.read.json("hdfs://namenode:9000/historico/json/lecturas.json")
dj.printSchema()
print(dj.count())
dj.select("sensor.zona","medicion.pm25","medicion.carros").groupBy("zona").count().show()

root
 |-- _id: struct (nullable = true)
 |    |-- $oid: string (nullable = true)
 |-- eventos: struct (nullable = true)
 |    |-- accidente: boolean (nullable = true)
 |    |-- es_fin_semana: boolean (nullable = true)
 |-- medicion: struct (nullable = true)
 |    |-- aqi: long (nullable = true)
 |    |-- carros: long (nullable = true)
 |    |-- co: double (nullable = true)
 |    |-- congestion_score: double (nullable = true)
 |    |-- emission_estimate: double (nullable = true)
 |    |-- fuel_waste_estimate: double (nullable = true)
 |    |-- no2: double (nullable = true)
 |    |-- o3: double (nullable = true)
 |    |-- pm10: double (nullable = true)
 |    |-- pm25: double (nullable = true)
 |    |-- temperatura: double (nullable = true)
 |-- sensor: struct (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- tipo: string (nullable = true)
 |    |-- ubicacion: struct (nullable = true)
 |    |    |-- lat: double (nullable = true)
 |    |    |-- lon: double (nullable = true

In [22]:
def a_pg(d, tabla):
    d.write.format("jdbc").option("url","jdbc:postgresql://postgres:5432/smartcity") \
     .option("dbtable",tabla).option("user","admin").option("password","admin123") \
     .option("driver","org.postgresql.Driver").mode("overwrite").save()

a_pg(mes, "resultado_spark_mensual")
a_pg(an.select("sensor_nombre","zona","fecha","carros","z"), "resultado_spark_anomalias")

In [23]:
import time
df.write.mode("overwrite").parquet("hdfs://namenode:9000/historico/parquet/datos")
t=time.time(); spark.read.csv("hdfs://namenode:9000/historico/datos_limpios.csv",header=True,inferSchema=True).groupBy("zona").count().collect(); print("CSV", time.time()-t)
t=time.time(); spark.read.parquet("hdfs://namenode:9000/historico/parquet/datos").groupBy("zona").count().collect(); print("Parquet", time.time()-t)

CSV 1.2650930881500244
Parquet 0.3954916000366211
